# Colocalization Analysis — 16 Sample Dataset


In [ ]:
import scanpy as sc
import squidpy as sq
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import norm
from scipy.spatial import cKDTree, Delaunay
import scipy.sparse as sp
import scipy.io as sio
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf
from scipy.stats import chi2_contingency
import pickle
import os
import gc
import warnings
warnings.filterwarnings("ignore")

OUTPUT_DIR = "colocalization_16_final"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/sample_level", exist_ok=True)


In [ ]:
# Data exported from Seurat using Matrix::writeMM to preserve all 18,934 genes.
# SeuratDisk::Convert() was not used as it silently drops non-HVGs.
#
# R export code:
#   library(Matrix)
#   writeMM(seurat_16sample[["RNA"]]$data, "normalized_counts.mtx")
#   write.csv(data.frame(gene = rownames(seurat_16sample[["RNA"]]$data)), "genes.csv", row.names = FALSE)
#   write.csv(data.frame(cell = colnames(seurat_16sample[["RNA"]]$data)), "barcodes.csv", row.names = FALSE)
#   write.csv(seurat_16sample@meta.data, "metadata.csv")

BUILD_FROM_MTX = False   # set True on first run to build from MTX files, False to reload h5ad

if BUILD_FROM_MTX:
    print("Building AnnData from MTX export...")
    X = sio.mmread("normalized_counts.mtx")
    X = sp.csr_matrix(X).T          # transpose to cells x genes
    genes    = pd.read_csv("genes.csv")
    barcodes = pd.read_csv("barcodes.csv")
    meta     = pd.read_csv("harmonized_seurat_metadata.csv", index_col=0)

    keep_mask = barcodes["cell"].isin(meta.index)
    keep_idx  = barcodes.index[keep_mask].values
    X_filtered       = X[keep_idx, :]
    barcodes_filtered = barcodes.loc[keep_mask].reset_index(drop=True)

    import anndata as ad
    adata16 = ad.AnnData(
        X   = X_filtered,
        obs = meta.loc[barcodes_filtered["cell"]],
        var = genes.set_index("gene"),
    )
    adata16.write_h5ad("seurat_16sample.h5ad")
    print("Saved: seurat_16sample.h5ad")

else:
    print("Loading from seurat_16sample.h5ad...")
    adata16 = sc.read_h5ad("seurat_16sample.h5ad")

print(f"Cells: {adata16.n_obs}, Genes: {adata16.n_vars}")


In [ ]:
# Map samples to response groups
responders = [
    "AGS18-05009", "AGS19-13043", "WPS17-15624", "WPS19-04204",
    "AGS21-16129", "AGS15-6075",  "AGS19-04330", "WPS19-02322",
]
non_responders = [
    "AGS18-00413", "AGS19-14722", "AGS20-09037", "SVS21-12350",
    "AGS17-11762", "AGS22-00104", "AGS17-18785", "AGS20-03666",
]

response_map = {s: "R" for s in responders}
response_map.update({s: "NR" for s in non_responders})
adata16.obs["response"] = adata16.obs["sample"].map(response_map)

adata16.obsm["spatial"] = adata16.obs[["x_slide_mm", "y_slide_mm"]].values
adata16.obs["sample"]   = adata16.obs["sample"].astype("category")
adata16.obs["response"] = adata16.obs["response"].astype("category")

print(f"R cells:  {(adata16.obs['response'] == 'R').sum()}")
print(f"NR cells: {(adata16.obs['response'] == 'NR').sum()}")
print(f"Unmapped: {adata16.obs['response'].isna().sum()}")


In [ ]:
IMMUNE_TYPES  = [
    "Macrophage_SPP1_Positive",
    "Macrophage_SPP1_Negative",
    "Lymphocytes",
    "Mast_cells",
    "Plasma",
]
ENDO_TYPES_ALL = ["Endothelial"]

sample_sizes = adata16.obs.groupby("sample").size()

def stouffer_z(z_scores, weights):
    w = np.array(weights)
    z = np.array(z_scores)
    mask = ~np.isnan(z)
    w, z = w[mask], z[mask]
    if len(z) == 0:
        return np.nan
    return np.sum(w * z) / np.sqrt(np.sum(w ** 2))

def run_sample_level_enrichment(adata, cluster_key, cell_types_of_interest,
                                 min_cells=5, n_perms=1000):
    samples = adata.obs["sample"].unique()
    records = []
    for sample in samples:
        adata_sub = adata[adata.obs["sample"] == sample].copy()
        adata_sub.obs[cluster_key] = adata_sub.obs[cluster_key].astype("category")
        ct_counts = adata_sub.obs[cluster_key].value_counts()
        missing = [ct for ct in cell_types_of_interest
                   if ct not in ct_counts.index or ct_counts[ct] < min_cells]
        if missing:
            print(f"  Skipping {sample} — insufficient cells: {missing}")
            continue
        sq.gr.spatial_neighbors(
            adata_sub, spatial_key="spatial",
            coord_type="generic", n_neighs=15,
        )
        sq.gr.nhood_enrichment(
            adata_sub, cluster_key=cluster_key,
            n_perms=n_perms, seed=42, show_progress_bar=False,
        )
        zscore_matrix = adata_sub.uns[f"{cluster_key}_nhood_enrichment"]["zscore"]
        labels   = adata_sub.obs[cluster_key].cat.categories.tolist()
        response = adata_sub.obs["response"].iloc[0]

        def get_z(ct1, ct2):
            if ct1 in labels and ct2 in labels:
                return zscore_matrix[labels.index(ct1), labels.index(ct2)]
            return np.nan

        record = {"sample": sample, "response": response}
        for immune in cell_types_of_interest:
            for endo in [ct for ct in cell_types_of_interest if "Endothelial" in ct]:
                record[f"{immune}_vs_{endo}"] = get_z(immune, endo)
        records.append(record)
    return pd.DataFrame(records)

def compute_stouffer_and_stats(df, score_cols, output_prefix):
    response_groups = sorted(df["response"].unique())
    g1, g2 = response_groups
    stouffer_rows = []
    stat_rows     = []
    for col in score_cols:
        if col not in df.columns:
            continue
        for resp in response_groups:
            sub = df[df["response"] == resp].dropna(subset=[col])
            weights = np.sqrt(sample_sizes.loc[sub["sample"]].values)
            z = stouffer_z(sub[col].values, weights)
            stouffer_rows.append({
                "comparison": col, "response": resp,
                "stouffer_z": z, "n_samples": len(sub),
                "median": sub[col].median(), "mean": sub[col].mean(),
            })
        s1 = df.loc[df["response"] == g1, col].dropna()
        s2 = df.loc[df["response"] == g2, col].dropna()
        if len(s1) > 1 and len(s2) > 1:
            u, p = stats.mannwhitneyu(s1, s2, alternative="two-sided")
            rbc  = 1 - (2 * u) / (len(s1) * len(s2))
        else:
            p, rbc = np.nan, np.nan
        stat_rows.append({
            "comparison": col,
            f"stouffer_z_{g1}": next((r["stouffer_z"] for r in stouffer_rows
                                      if r["comparison"]==col and r["response"]==g1), np.nan),
            f"stouffer_z_{g2}": next((r["stouffer_z"] for r in stouffer_rows
                                      if r["comparison"]==col and r["response"]==g2), np.nan),
            f"n_{g1}": len(s1), f"n_{g2}": len(s2),
            f"median_{g1}": s1.median() if len(s1) > 0 else np.nan,
            f"median_{g2}": s2.median() if len(s2) > 0 else np.nan,
            "MWU_p": p, "rank_biserial_r": rbc,
        })
    df_stouffer = pd.DataFrame(stouffer_rows)
    df_stats    = pd.DataFrame(stat_rows)
    valid = df_stats["MWU_p"].notna()
    if valid.sum() > 0:
        _, p_adj, _, _ = multipletests(df_stats.loc[valid, "MWU_p"], method="fdr_bh")
        df_stats.loc[valid, "MWU_p_adj"] = p_adj
    df_stouffer.to_csv(f"{OUTPUT_DIR}/sample_level/{output_prefix}_stouffer.csv", index=False)
    df_stats.to_csv(f"{OUTPUT_DIR}/sample_level/{output_prefix}_stats.csv", index=False)
    return df_stouffer, df_stats

# Run enrichment — all Endothelial only
print("Running per-sample neighborhood enrichment...")
adata16.obs["cluster_for_cellchat"] = adata16.obs["cluster_for_cellchat"].astype("category")

df_sample_simple = run_sample_level_enrichment(
    adata16,
    cluster_key="cluster_for_cellchat",
    cell_types_of_interest=IMMUNE_TYPES + ENDO_TYPES_ALL,
)
df_sample_simple.to_csv(
    f"{OUTPUT_DIR}/sample_level/sample_enrichment_all_endo.csv", index=False
)

score_cols_simple = [f"{immune}_vs_Endothelial" for immune in IMMUNE_TYPES]
df_stouffer_simple, df_stats_simple = compute_stouffer_and_stats(
    df_sample_simple, score_cols_simple, "all_endo"
)

In [ ]:
# Convert Stouffer Z scores to plot dataframe

def z_to_p(z):
    return 2 * (1 - norm.cdf(abs(z)))

plot_rows = []
for _, row in df_stouffer_simple.iterrows():
    z = row["stouffer_z"]
    if np.isnan(z):
        continue
    p = z_to_p(z)
    plot_rows.append({
        "comparison":  row["comparison"],
        "response":    row["response"],
        "stouffer_z":  z,
        "p":           p,
        "direction":   "enriched" if z > 0 else "depleted",
        "significant": p < 0.05,
    })

df_plot = pd.DataFrame(plot_rows)
df_plot["immune_type"] = df_plot["comparison"].str.replace(
    "_vs_Endothelial", "", regex=False
)

response_order = sorted(df_plot["response"].unique())
immune_order   = IMMUNE_TYPES[::-1]


## Colocalization with Endothelial Cells

Stouffer's weighted Z-score per response group. Red = enriched (Z > 1.96), blue = depleted (Z < −1.96), gray = not significant.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 7))

DOT_SIZE = 200

for _, row in df_plot.iterrows():
    x = response_order.index(row["response"])
    y = immune_order.index(row["immune_type"]) \
        if row["immune_type"] in immune_order else None
    if y is None:
        continue

    # Color by Z threshold (1.96 = p~0.05 for standard normal)
    if row["stouffer_z"] > 1.96:
        color = "#C62828"
    elif row["stouffer_z"] < -1.96:
        color = "#1565C0"
    else:
        color = "lightgray"

    ax.scatter(x, y, s=DOT_SIZE, color=color,
               edgecolors="black", linewidths=0.8, zorder=2)

ax.set_xticks(range(len(response_order)))
ax.set_xticklabels(response_order, fontsize=11)
ax.set_yticks(range(len(immune_order)))
ax.set_yticklabels(immune_order, fontsize=9)
ax.set_xlim(-0.5, len(response_order) - 0.5)
ax.set_ylim(-0.5, len(immune_order) - 0.5)
ax.grid(True, color="lightgray", linewidth=0.5, zorder=0)
ax.set_title("Colocalization with Endothelial Cells\n"
             "Stouffer's Weighted Z (|Z|>1.96 colored)",
             fontsize=11)

color_handles = [
    mpatches.Patch(facecolor="#C62828", label="Enriched (Z>1.96)"),
    mpatches.Patch(facecolor="#1565C0", label="Depleted (Z<-1.96)"),
    mpatches.Patch(facecolor="lightgray", label="Not significant"),
]
ax.legend(handles=color_handles, loc="upper left",
          bbox_to_anchor=(1.02, 1), fontsize=9, title_fontsize=10,
          title="Direction")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/sample_level/dotplot_all_endo.png",
            dpi=150, bbox_inches="tight")
plt.savefig(f"{OUTPUT_DIR}/sample_level/dotplot_all_endo.pdf",
            bbox_inches="tight")
plt.show()
print("Saved: dotplot_all_endo.png/.pdf")

## AGT Expression in Macrophages

AGT expression in SPP1+ and SPP1- Macrophages, NR vs R. Normalized from raw counts. Bar plot shows per-sample mean ± SEM with Mann-Whitney p-values.

In [ ]:
# AGT / REN expression in SPP1+ and SPP1- Macrophages — NR vs R
# Per-sample mean ± SEM bar plot
# Normalized from raw counts: normalize_total(1e4) + log1p

import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np
import scipy.sparse as sp
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

GENES_OF_INTEREST = ["AGT", "REN"]
MAC_TYPES         = ["Macrophage_SPP1_Positive", "Macrophage_SPP1_Negative"]
OUTPUT_DIR        = "colocalization_16"

genes_found = [g for g in GENES_OF_INTEREST if g in adata16.var_names]
print(f"Genes found: {genes_found}")

# ── Normalize and build expr_df ───────────────────────────────────────────────
mac_mask  = adata16.obs["cluster_for_cellchat"].isin(MAC_TYPES)
adata_mac = adata16[mac_mask].copy()

sc.pp.normalize_total(adata_mac, target_sum=1e4)
sc.pp.log1p(adata_mac)

expr_df = pd.DataFrame({
    "cluster_for_cellchat": adata_mac.obs["cluster_for_cellchat"].values.astype(str),
    "sample":               adata_mac.obs["sample"].values.astype(str),
    "response":             adata_mac.obs["response"].values.astype(str),
})

for gene in genes_found:
    idx = list(adata_mac.var_names).index(gene)
    X   = adata_mac.X[:, idx]
    expr_df[gene] = (
        X.toarray().flatten() if sp.issparse(X) else np.array(X).flatten()
    )

response_groups = sorted(expr_df["response"].unique())
g1, g2 = response_groups
print(f"Response groups: g1={g1}, g2={g2}")

GROUP_ORDER = [
    f"SPP1 Positive {g1}",
    f"SPP1 Positive {g2}",
    f"SPP1 Negative {g1}",
    f"SPP1 Negative {g2}",
]
GROUP_PALETTE = {
    f"SPP1 Positive {g1}": "#B71C1C",
    f"SPP1 Positive {g2}": "#EF9A9A",
    f"SPP1 Negative {g1}": "#0D47A1",
    f"SPP1 Negative {g2}": "#90CAF9",
}

# ── Bar plot: per-sample mean ± SEM ──────────────────────────────────────────
n_genes = len(genes_found)
fig, axes = plt.subplots(1, n_genes, figsize=(5.5 * n_genes, 6), squeeze=False)
fig.suptitle("AGT / REN Expression in Macrophages — Per-sample mean ± SEM", fontsize=13)

for col_i, gene in enumerate(genes_found):
    ax = axes[0][col_i]

    # Per-sample means
    sample_rows = []
    for (sample, response, mac_type), grp in expr_df.groupby(
        ["sample", "response", "cluster_for_cellchat"]
    ):
        mac_label = mac_type.replace("Macrophage_SPP1_", "SPP1 ")
        sample_rows.append({
            "group":  f"{mac_label} {response}",
            "sample": sample,
            "mean":   grp[gene].mean(),
        })
    sample_df = pd.DataFrame(sample_rows)

    summary = sample_df.groupby("group")["mean"].agg(
        mean_val="mean",
        sem_val=lambda x: x.std() / np.sqrt(len(x)),
        n="count",
    ).reset_index()
    summary["group"] = pd.Categorical(
        summary["group"], categories=GROUP_ORDER, ordered=True
    )
    summary = summary.sort_values("group").reset_index(drop=True)

    colors = [GROUP_PALETTE[g] for g in summary["group"]]
    x      = np.arange(len(summary))

    ax.bar(x, summary["mean_val"], color=colors,
           edgecolor="black", linewidth=0.7, width=0.6)
    ax.errorbar(x, summary["mean_val"], yerr=summary["sem_val"],
                fmt="none", color="black", capsize=4, linewidth=1.2, zorder=3)

    ax.set_ylim(bottom=0)
    ax.set_xticks(x)
    ax.set_xticklabels(
        [f"SPP1+\n{g1}", f"SPP1+\n{g2}", f"SPP1-\n{g1}", f"SPP1-\n{g2}"],
        fontsize=9
    )
    ax.set_xlabel("")
    ax.set_ylabel("Mean expression (log-normalized)" if col_i == 0 else "")
    ax.set_title(gene, fontsize=12)
    ax.axvline(1.5, color="gray", linestyle="--", linewidth=0.8, alpha=0.7)

    # P-value brackets from per-sample MWU
    for mac_i, mac_type in enumerate([
        "Macrophage_SPP1_Positive",
        "Macrophage_SPP1_Negative",
    ]):
        mac_label = mac_type.replace("Macrophage_SPP1_", "SPP1 ")
        s_nr = sample_df.loc[sample_df["group"] == f"{mac_label} {g1}", "mean"]
        s_r  = sample_df.loc[sample_df["group"] == f"{mac_label} {g2}", "mean"]
        if len(s_nr) > 1 and len(s_r) > 1:
            _, p  = stats.mannwhitneyu(s_nr, s_r, alternative="two-sided")
            pstr  = f"p={p:.3f}" if p >= 0.001 else f"p={p:.2e}"
            color = "black" if p < 0.05 else "gray"
        else:
            pstr, color = "NA", "gray"

        x1, x2 = mac_i * 2, mac_i * 2 + 1
        y_max = max(
            summary.iloc[x1]["mean_val"] + summary.iloc[x1]["sem_val"],
            summary.iloc[x2]["mean_val"] + summary.iloc[x2]["sem_val"],
        )
        h = ax.get_ylim()[1] * 0.04
        ax.plot([x1, x1, x2, x2],
                [y_max + h, y_max + 2*h, y_max + 2*h, y_max + h],
                color="black", linewidth=0.8)
        ax.text((x1 + x2) / 2, y_max + 2.5*h, pstr,
                ha="center", fontsize=8, style="italic", color=color)

legend_elements = [
    mpatches.Patch(facecolor="#B71C1C", label=f"SPP1+ {g1}"),
    mpatches.Patch(facecolor="#EF9A9A", label=f"SPP1+ {g2}"),
    mpatches.Patch(facecolor="#0D47A1", label=f"SPP1- {g1}"),
    mpatches.Patch(facecolor="#90CAF9", label=f"SPP1- {g2}"),
]
fig.legend(handles=legend_elements, loc="lower center",
           ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.01))
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig(f"{OUTPUT_DIR}/angiotensin_barplot_per_sample.png",
            dpi=150, bbox_inches="tight")
plt.savefig(f"{OUTPUT_DIR}/angiotensin_barplot_per_sample.pdf",
            bbox_inches="tight")
plt.show()
plt.close()
print("Saved: angiotensin_barplot_per_sample.png/.pdf")

In [ ]:
# Spatial neighbor graph (per-sample 15-nearest-neighbors).
# Built per-sample on slide coordinates so neighbors never cross tissue sections.
# Each cell maps to its 15 nearest neighbors, keyed by global index into
# adata16, with the cell itself excluded.

os.makedirs(f"{OUTPUT_DIR}/neighbors", exist_ok=True)

BUILD_NEIGHBORS = False   # set True to build from scratch, False to reload pkl
NEIGHBOR_PKL    = f"{OUTPUT_DIR}/neighbors/neighbor_graphs.pkl"

def build_knn_dict(adata, k):
    """Per-sample KNN neighbor dictionary, keyed by global index (self excluded)."""
    samples = adata.obs["sample"].values
    coords  = adata.obsm["spatial"]
    neighbor_dict = {}
    for sample in np.unique(samples):
        sample_idx = np.where(samples == sample)[0]
        if len(sample_idx) <= k:
            continue
        tree = cKDTree(coords[sample_idx])
        _, nn_local = tree.query(coords[sample_idx], k=k + 1)
        for local_i, row in enumerate(nn_local):
            global_i = sample_idx[local_i]
            neighbor_dict[global_i] = sample_idx[row[1:]]   # row[0] is self
    return neighbor_dict

if BUILD_NEIGHBORS:
    print("Building knn15 spatial neighbor graph...")
    neighbor_graphs = {"knn15": build_knn_dict(adata16, 15)}
    n_neighbors = [len(v) for v in neighbor_graphs["knn15"].values()]
    print(f"  Median neighbors: {np.median(n_neighbors):.1f}, "
          f"Mean: {np.mean(n_neighbors):.1f}")
    with open(NEIGHBOR_PKL, "wb") as f:
        pickle.dump(neighbor_graphs, f)
    print(f"Saved: {NEIGHBOR_PKL}")
else:
    print(f"Loading neighbor graph from {NEIGHBOR_PKL}...")
    with open(NEIGHBOR_PKL, "rb") as f:
        neighbor_graphs = pickle.load(f)

knn15 = neighbor_graphs["knn15"]
print(f"knn15 cells: {len(knn15)}")

## ICAM1 / MADCAM1 Expression Around SPP1+ AGT+ Macrophages

Log fold-change of ICAM1 and MADCAM1 in cells neighboring SPP1+ AGT+ macrophages, relative to each sample's whole-sample expression baseline (NR vs R).

In [ ]:
# ICAM1 / MADCAM1 log fold-change vs whole-sample baseline,
# in cells neighboring SPP1+ AGT+ Macrophages.
#
# Baseline = mean expression across ALL cells in the sample (log-norm space).
# log FC   = neighbor cell expression - sample baseline.
# Aggregated to per-sample pseudobulk (pb_df) for the NR vs R bar plot below.

import scanpy as sc
import scipy.sparse as sp

GENES = ["ICAM1", "MADCAM1"]

# -- Normalize all cells -------------------------------------------------------
print("Normalizing all cells...")
adata_norm = adata16.copy()
sc.pp.normalize_total(adata_norm, target_sum=1e4)
sc.pp.log1p(adata_norm)

def get_expr(adata, gene):
    idx = list(adata.var_names).index(gene)
    X   = adata.X[:, idx]
    return X.toarray().flatten() if sp.issparse(X) else np.array(X).flatten()

all_df = pd.DataFrame({
    "sample":    adata16.obs["sample"].astype(str).values,
    "response":  adata16.obs["response"].astype(str).values,
    "cell_type": adata16.obs["cluster_for_cellchat"].astype(str).values,
})
for gene in GENES:
    all_df[gene] = get_expr(adata_norm, gene)

del adata_norm
gc.collect()

response_groups = sorted(all_df["response"].unique())
g1, g2 = response_groups
print(f"g1={g1}, g2={g2}")

# -- Per-sample whole-sample baseline (mean across all cells) ------------------
baseline_rows = []
for (sample, response), grp in all_df.groupby(["sample", "response"]):
    row = {"sample": sample, "response": response, "n_cells": len(grp)}
    for gene in GENES:
        row[f"baseline_{gene}"] = grp[gene].mean()
    baseline_rows.append(row)
baseline_df = pd.DataFrame(baseline_rows).sort_values(["response", "sample"])

baseline_maps = {
    gene: dict(zip(baseline_df["sample"], baseline_df[f"baseline_{gene}"]))
    for gene in GENES
}

# -- Identify SPP1+ AGT+ macrophages and their neighbors -----------------------
spp1pos_mask   = adata16.obs["cluster_for_cellchat"] == "Macrophage_SPP1_Positive"
spp1pos_global = np.where(spp1pos_mask.values)[0]

adata_spp1 = adata16[spp1pos_mask].copy()
sc.pp.normalize_total(adata_spp1, target_sum=1e4)
sc.pp.log1p(adata_spp1)
agt_idx  = list(adata_spp1.var_names).index("AGT")
X_agt    = adata_spp1.X[:, agt_idx]
agt_expr = X_agt.toarray().flatten() if sp.issparse(X_agt) else np.array(X_agt).flatten()
agt_mac_global = spp1pos_global[agt_expr > 0]
del adata_spp1
gc.collect()
print(f"SPP1+ AGT+ macs: {len(agt_mac_global)}")

neighbor_set = set()
for global_i in agt_mac_global:
    neighbors = knn15.get(global_i, np.array([], dtype=int))
    neighbor_set.update(neighbors.tolist())
neighbor_set -= set(agt_mac_global.tolist())
neighbor_idx = np.array(sorted(neighbor_set))
print(f"Unique neighbor cells: {len(neighbor_idx)}")

# -- Neighbor log FC per gene --------------------------------------------------
nbr_df = pd.DataFrame({
    "sample":    all_df["sample"].iloc[neighbor_idx].values,
    "response":  all_df["response"].iloc[neighbor_idx].values,
    "cell_type": all_df["cell_type"].iloc[neighbor_idx].values,
})
for gene in GENES:
    nbr_df[gene]               = all_df[gene].iloc[neighbor_idx].values
    nbr_df[f"baseline_{gene}"] = nbr_df["sample"].map(baseline_maps[gene])
    nbr_df[f"logFC_{gene}"]    = nbr_df[gene] - nbr_df[f"baseline_{gene}"]

# -- Per-sample pseudobulk -----------------------------------------------------
pb_rows = []
for (sample, response), grp in nbr_df.groupby(["sample", "response"]):
    row = {"sample": sample, "response": response, "n_cells": len(grp)}
    for gene in GENES:
        row[f"mean_logFC_{gene}"] = grp[f"logFC_{gene}"].mean()
    pb_rows.append(row)
pb_df = pd.DataFrame(pb_rows).sort_values(["response", "sample"])

pb_df.to_csv(
    f"{OUTPUT_DIR}/ICAM1_MADCAM1_allcell_baseline_logFC_pseudobulk.csv",
    index=False,
)
baseline_df.to_csv(
    f"{OUTPUT_DIR}/ICAM1_MADCAM1_allcell_baseline_per_sample.csv",
    index=False,
)
print("Saved: pseudobulk + baseline CSVs")

In [ ]:
# =============================================================================
# Final bar plot: ICAM1 and MADCAM1 log FC vs whole-sample baseline
# X axis: NR ICAM1 | NR MADCAM1 | R ICAM1 | R MADCAM1
# Error bars = SEM, p-value = Wilcoxon signed-rank vs 0
# =============================================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

OUTPUT_DIR = "colocalization_16"

# -- Palette: NR = reds, R = blues, ICAM1 = dark, MADCAM1 = light --------------
BAR_PALETTE = {
    (g1, "ICAM1"):   "#B71C1C",
    (g1, "MADCAM1"): "#EF9A9A",
    (g2, "ICAM1"):   "#0D47A1",
    (g2, "MADCAM1"): "#90CAF9",
}
# X positions with a gap between NR and R
X_POS = {
    (g1, "ICAM1"):   0,
    (g1, "MADCAM1"): 0.6,
    (g2, "ICAM1"):   1.6,
    (g2, "MADCAM1"): 2.2,
}

fig, ax = plt.subplots(figsize=(7, 5))
fig.suptitle(
    "ICAM1 / MADCAM1 log FC vs whole-sample baseline\n"
    "Cells neighboring SPP1+ AGT+ Macrophages",
    fontsize=11
)

for (resp, gene), xi in X_POS.items():
    vals  = pb_df.loc[pb_df["response"] == resp,
                      f"mean_logFC_{gene}"].dropna()
    color = BAR_PALETTE[(resp, gene)]
    mean  = vals.mean()
    sem   = vals.std() / np.sqrt(len(vals))
    ax.bar(xi, mean, width=0.5, color=color, alpha=0.85,
           edgecolor="black", linewidth=0.8)
    ax.errorbar(xi, mean, yerr=sem,
                fmt="none", color="black",
                capsize=5, linewidth=1.5, zorder=3)

    # Wilcoxon signed-rank vs 0
    try:
        _, p_w = stats.wilcoxon(vals, zero_method="wilcox",
                                 alternative="two-sided")
    except Exception:
        p_w = np.nan
    pstr  = f"p={p_w:.3f}" if (not np.isnan(p_w) and p_w >= 0.001) \
            else (f"p={p_w:.2e}" if not np.isnan(p_w) else "NA")
    color_p = "black" if (not np.isnan(p_w) and p_w < 0.05) else "gray"
    y_top = mean + sem + abs(ax.get_ylim()[1]) * 0.03
    ax.text(xi, y_top, pstr,
            ha="center", va="bottom", fontsize=8,
            style="italic", color=color_p)

ax.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.6)

# X axis
ax.set_xticks(list(X_POS.values()))
ax.set_xticklabels(
    [f"{resp}\n{gene}" for resp, gene in X_POS.keys()],
    fontsize=9
)

# Separator between NR and R
ax.axvline(1.1, color="gray", linestyle="--", linewidth=0.8, alpha=0.7)

ylim = ax.get_ylim()
ax.axhspan(0, ylim[1], alpha=0.03, color="#C62828", zorder=0)
ax.axhspan(ylim[0], 0, alpha=0.03, color="#1565C0", zorder=0)
ax.set_ylim(ylim)
ax.set_ylabel("Mean log FC (vs whole-sample baseline)\nper-sample pseudobulk ± SEM")
ax.set_xlabel("")
ax.text(0.02, 0.97, "Above baseline", transform=ax.transAxes,
        fontsize=7, va="top", color="#C62828", alpha=0.7)
ax.text(0.02, 0.03, "Below baseline", transform=ax.transAxes,
        fontsize=7, va="bottom", color="#1565C0", alpha=0.7)

legend_elements = [
    mpatches.Patch(facecolor="#B71C1C", label=f"ICAM1 {g1}"),
    mpatches.Patch(facecolor="#EF9A9A", label=f"MADCAM1 {g1}"),
    mpatches.Patch(facecolor="#0D47A1", label=f"ICAM1 {g2}"),
    mpatches.Patch(facecolor="#90CAF9", label=f"MADCAM1 {g2}"),
    plt.Line2D([0], [0], color="black", linewidth=0.8,
               linestyle="--", label="Baseline (log FC = 0)"),
]
ax.legend(handles=legend_elements, fontsize=8,
          loc="upper right", framealpha=0.9)

plt.tight_layout()
plt.savefig(
    f"{OUTPUT_DIR}/ICAM1_MADCAM1_logFC_final_barplot.png",
    dpi=150, bbox_inches="tight")
plt.savefig(
    f"{OUTPUT_DIR}/ICAM1_MADCAM1_logFC_final_barplot.pdf",
    bbox_inches="tight")
plt.show()
plt.close()
print("Saved: ICAM1_MADCAM1_logFC_final_barplot.png/.pdf")

## SPP1+ AGT+ Macrophages with CTSD+ Neighbors

Per-sample percentage of SPP1+ macrophages that are AGT+ and have at least one CTSD+ cell in their spatial neighborhood (NR vs R). Tested with per-sample Mann-Whitney and a cell-level binomial GLMM (sample as random effect).

In [ ]:
# =============================================================================
# % SPP1+ Macrophages that are AGT+ AND have CTSD+ in their neighborhood
# NR vs R — per-sample percentage, MWU + binomial GLMM
# =============================================================================

import scanpy as sc
import pandas as pd
import numpy as np
import scipy.sparse as sp
import scipy.stats as stats
import statsmodels.formula.api as smf
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM
from scipy.stats import norm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pickle, gc
import warnings
warnings.filterwarnings("ignore")

OUTPUT_DIR = "colocalization_16"

response_groups = sorted(adata16.obs["response"].astype(str).unique())
g1, g2 = response_groups

# -- Load neighbor graph -------------------------------------------------------
with open(f"{OUTPUT_DIR}/neighbors/neighbor_graphs.pkl", "rb") as f:
    neighbor_graphs = pickle.load(f)
knn15 = neighbor_graphs["knn15"]

# -- Normalize all cells - extract CTSD ----------------------------------------
adata_all = adata16.copy()
sc.pp.normalize_total(adata_all, target_sum=1e4)
sc.pp.log1p(adata_all)

ctsd_idx = list(adata_all.var_names).index("CTSD")
X_ctsd   = adata_all.X[:, ctsd_idx]
ctsd_all = X_ctsd.toarray().flatten() if sp.issparse(X_ctsd) \
           else np.array(X_ctsd).flatten()

del adata_all
gc.collect()

# -- Normalize SPP1+ macs - extract AGT ----------------------------------------
spp1pos_mask   = adata16.obs["cluster_for_cellchat"] == "Macrophage_SPP1_Positive"
spp1pos_global = np.where(spp1pos_mask.values)[0]

adata_spp1 = adata16[spp1pos_mask].copy()
sc.pp.normalize_total(adata_spp1, target_sum=1e4)
sc.pp.log1p(adata_spp1)

agt_idx  = list(adata_spp1.var_names).index("AGT")
X_agt    = adata_spp1.X[:, agt_idx]
agt_expr = X_agt.toarray().flatten() if sp.issparse(X_agt) \
           else np.array(X_agt).flatten()

mac_df = pd.DataFrame({
    "sample":   adata_spp1.obs["sample"].values.astype(str),
    "response": adata_spp1.obs["response"].values.astype(str),
    "AGT_pos":  (agt_expr > 0).astype(int),
})

del adata_spp1
gc.collect()

# -- CTSD spatial lag: does any neighbor express CTSD? -------------------------
ctsd_lag_pos = np.zeros(len(spp1pos_global), dtype=int)
for local_i, global_i in enumerate(spp1pos_global):
    neighbors = knn15.get(global_i, np.array([], dtype=int))
    if len(neighbors) > 0:
        ctsd_lag_pos[local_i] = int((ctsd_all[neighbors] > 0).any())

mac_df["ctsd_lag_pos"] = ctsd_lag_pos
mac_df["combo_pos"]    = (
    (mac_df["AGT_pos"] == 1) & (mac_df["ctsd_lag_pos"] == 1)
).astype(int)

# -- Per-sample % AGT+ with CTSD+ neighbor -------------------------------------
pb_rows = []
for (sample, response), grp in mac_df.groupby(["sample", "response"]):
    pb_rows.append({
        "sample":   sample,
        "response": response,
        "pct":      grp["combo_pos"].mean() * 100,
    })
pb_df = pd.DataFrame(pb_rows)

s1 = pb_df.loc[pb_df["response"] == g1, "pct"]
s2 = pb_df.loc[pb_df["response"] == g2, "pct"]
_, p_mwu = stats.mannwhitneyu(s1, s2, alternative="two-sided")

# -- Binomial GLMM (logit link) on the cell-level binary outcome ---------------
# statsmodels MixedLM has no binomial family, so use BinomialBayesMixedGLM;
# equivalent to glmer(combo_pos ~ response + (1|sample), family = binomial).
mac_df["response_bin"] = (mac_df["response"] == g1).astype(int)

try:
    glmm = BinomialBayesMixedGLM.from_formula(
        "combo_pos ~ response_bin",
        {"sample": "0 + C(sample)"},
        data=mac_df
    ).fit_map()

    cov_names = glmm.model.fep_names      # fixed effect parameter names
    resp_idx  = cov_names.index("response_bin")
    coef_log  = glmm.params[resp_idx]     # log-odds scale
    se_log    = glmm.fe_sd[resp_idx]      # SE on log-odds scale
    z_stat    = coef_log / se_log
    p_glmm    = 2 * (1 - norm.cdf(abs(z_stat)))
    or_val    = np.exp(coef_log)          # odds ratio
    or_lo     = np.exp(coef_log - 1.96 * se_log)
    or_hi     = np.exp(coef_log + 1.96 * se_log)

    print(f"GLMM (binomial, logit link):")
    print(f"  OR={or_val:.3f} (95% CI: {or_lo:.3f}, {or_hi:.3f})")
    print(f"  log-OR={coef_log:.4f}, SE={se_log:.4f}, z={z_stat:.3f}, p={p_glmm:.4f}")
    print(f"  Interpretation: {g1} cells have "
          f"{'higher' if coef_log > 0 else 'lower'} odds of being "
          f"AGT+ with CTSD+ neighbor (OR={or_val:.3f}) vs {g2}")

except Exception as e:
    print(f"  GLMM failed: {e}")
    p_glmm = np.nan
    or_val = np.nan
    print("  Falling back to LMM approximation...")
    try:
        model  = smf.mixedlm("combo_pos ~ response_bin",
                              data=mac_df,
                              groups=mac_df["sample"]).fit(reml=True)
        p_glmm = model.pvalues["response_bin"]
        coef   = model.params["response_bin"]
        print(f"  LMM fallback: \u03b2={coef:.4f}, p={p_glmm:.4f}")
    except Exception as e2:
        print(f"  LMM fallback also failed: {e2}")

# -- Plot ----------------------------------------------------------------------
PALETTE = {g1: "#C62828", g2: "#1565C0"}

fig, ax = plt.subplots(figsize=(4, 5))
fig.suptitle(
    "SPP1+ Macrophages\nAGT+ with CTSD+ neighbor - NR vs R",
    fontsize=11
)

for xi, resp in enumerate([g1, g2]):
    vals  = pb_df.loc[pb_df["response"] == resp, "pct"]
    color = PALETTE[resp]
    ax.bar(xi, vals.mean(), color=color, alpha=0.7,
           edgecolor="black", linewidth=0.8, width=0.5)
    ax.errorbar(xi, vals.mean(),
                yerr=vals.std() / np.sqrt(len(vals)),
                fmt="none", color="black",
                capsize=5, linewidth=1.5, zorder=3)

ax.set_ylim(bottom=0)
y_top = ax.get_ylim()[1]

# MWU bracket
ax.plot([0, 0, 1, 1],
        [y_top*0.82, y_top*0.86, y_top*0.86, y_top*0.82],
        color="black", linewidth=0.8)
pstr_mwu = f"p={p_mwu:.3f}" if p_mwu >= 0.001 else f"p={p_mwu:.2e}"
ax.text(0.5, y_top*0.87, f"MWU {pstr_mwu}",
        ha="center", fontsize=8, style="italic",
        color="black" if p_mwu < 0.05 else "gray")

# GLMM annotation
if not np.isnan(p_glmm):
    pstr_glmm = f"p={p_glmm:.3f}" if p_glmm >= 0.001 else f"p={p_glmm:.2e}"
    or_str    = f"OR={or_val:.2f}" if not np.isnan(or_val) else ""
    ax.text(0.5, y_top * 0.95,
            f"GLMM {pstr_glmm} ({or_str})",
            ha="center", fontsize=8, style="italic",
            color="black" if p_glmm < 0.05 else "gray")

legend_elements = [
    mpatches.Patch(facecolor=PALETTE[g1], alpha=0.7, label=g1),
    mpatches.Patch(facecolor=PALETTE[g2], alpha=0.7, label=g2),
]
fig.legend(handles=legend_elements, loc="lower center",
           ncol=2, fontsize=9, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig(f"{OUTPUT_DIR}/AGT_CTSD_combo_pct_NR_vs_R.png",
            dpi=150, bbox_inches="tight")
plt.savefig(f"{OUTPUT_DIR}/AGT_CTSD_combo_pct_NR_vs_R.pdf",
            bbox_inches="tight")
plt.show()
plt.close()
print("Saved: AGT_CTSD_combo_pct_NR_vs_R.png/.pdf")

## ICAM1 / MADCAM1 Expression in Endothelial Cells

ICAM1 and MADCAM1 expression across all endothelial cells (per-sample mean ± SEM, NR vs R), with paired Wilcoxon comparing ICAM1 vs MADCAM1 within each response group.

In [ ]:
# =============================================================================
# Bar plot: ICAM1 and MADCAM1 expression in ALL endothelial cells
# X axis: NR ICAM1 | NR MADCAM1 | R ICAM1 | R MADCAM1
# P-values: paired Wilcoxon ICAM1 vs MADCAM1 within NR and within R
# =============================================================================

import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np
import scipy.sparse as sp
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

OUTPUT_DIR = "colocalization_16"

# -- Normalize endothelial cells -----------------------------------------------
endo_mask  = adata16.obs["cluster_for_cellchat"] == "Endothelial"
adata_endo = adata16[endo_mask].copy()
sc.pp.normalize_total(adata_endo, target_sum=1e4)
sc.pp.log1p(adata_endo)

def get_expr(adata, gene):
    idx = list(adata.var_names).index(gene)
    X   = adata.X[:, idx]
    return X.toarray().flatten() if sp.issparse(X) else np.array(X).flatten()

endo_df = pd.DataFrame({
    "sample":   adata_endo.obs["sample"].astype(str).values,
    "response": adata_endo.obs["response"].astype(str).values,
    "ICAM1":    get_expr(adata_endo, "ICAM1"),
    "MADCAM1":  get_expr(adata_endo, "MADCAM1"),
})

response_groups = sorted(endo_df["response"].unique())
g1, g2 = response_groups
print(f"g1={g1}, g2={g2}")
print(f"Endothelial cells: {len(endo_df)}")
print(endo_df.groupby("response")[["ICAM1", "MADCAM1"]].mean().round(4))

# -- Per-sample means ----------------------------------------------------------
pb_rows = []
for (sample, response), grp in endo_df.groupby(["sample", "response"]):
    pb_rows.append({
        "sample":   sample,
        "response": response,
        "ICAM1":    grp["ICAM1"].mean(),
        "MADCAM1":  grp["MADCAM1"].mean(),
        "n_cells":  len(grp),
    })
pb_endo = pd.DataFrame(pb_rows).sort_values(["response", "sample"])

print(f"\nPer-sample means:")
print(pb_endo.to_string(index=False))

# -- Plot setup ----------------------------------------------------------------
BAR_PALETTE = {
    (g1, "ICAM1"):   "#B71C1C",
    (g1, "MADCAM1"): "#EF9A9A",
    (g2, "ICAM1"):   "#0D47A1",
    (g2, "MADCAM1"): "#90CAF9",
}
X_POS = {
    (g1, "ICAM1"):   0,
    (g1, "MADCAM1"): 0.6,
    (g2, "ICAM1"):   1.6,
    (g2, "MADCAM1"): 2.2,
}

fig, ax = plt.subplots(figsize=(7, 6))
fig.suptitle(
    "ICAM1 and MADCAM1 expression in Endothelial Cells\n"
    "Per-sample mean ± SEM — NR vs R",
    fontsize=11
)

# -- Draw bars -----------------------------------------------------------------
bar_stats = {}
for (resp, gene), xi in X_POS.items():
    vals  = pb_endo.loc[pb_endo["response"] == resp, gene].dropna()
    color = BAR_PALETTE[(resp, gene)]
    mean  = vals.mean()
    sem   = vals.std() / np.sqrt(len(vals))

    ax.bar(xi, mean, width=0.5, color=color, alpha=0.85,
           edgecolor="black", linewidth=0.8)
    ax.errorbar(xi, mean, yerr=sem,
                fmt="none", color="black",
                capsize=5, linewidth=1.5, zorder=3)

    bar_stats[(resp, gene)] = {"mean": mean, "sem": sem, "vals": vals}

# -- Set y headroom for brackets -----------------------------------------------
all_tops = [v["mean"] + v["sem"] for v in bar_stats.values()]
y_range  = max(all_tops) - 0
ax.set_ylim(
    bottom=0,
    top=max(all_tops) + y_range * 0.5
)

# -- Brackets: paired Wilcoxon ICAM1 vs MADCAM1 within NR and within R ---------
bracket_y = max(all_tops) + y_range * 0.08

for resp in [g1, g2]:
    icam_series = pb_endo.loc[
        pb_endo["response"] == resp, ["sample", "ICAM1"]
    ].set_index("sample")["ICAM1"]
    mad_series  = pb_endo.loc[
        pb_endo["response"] == resp, ["sample", "MADCAM1"]
    ].set_index("sample")["MADCAM1"]
    common = icam_series.index.intersection(mad_series.index)
    diff   = icam_series.loc[common] - mad_series.loc[common]

    print(f"\n{resp} — ICAM1 vs MADCAM1 per sample:")
    print(f"  ICAM1 mean:   {icam_series.mean():.4f}")
    print(f"  MADCAM1 mean: {mad_series.mean():.4f}")
    print(f"  Differences:  {diff.round(4).tolist()}")

    try:
        _, p_pair = stats.wilcoxon(diff, zero_method="wilcox",
                                    alternative="two-sided")
        pstr  = f"p={p_pair:.3f}" if p_pair >= 0.001 else f"p={p_pair:.2e}"
        color = "black" if p_pair < 0.05 else "gray"
        print(f"  Paired Wilcoxon: {pstr}")
    except Exception as e:
        pstr, color = "NA", "gray"
        print(f"  Paired Wilcoxon failed: {e}")

    x_icam = X_POS[(resp, "ICAM1")]
    x_mad  = X_POS[(resp, "MADCAM1")]
    h      = y_range * 0.04

    ax.plot([x_icam, x_icam, x_mad, x_mad],
            [bracket_y, bracket_y + h,
             bracket_y + h, bracket_y],
            color="black", linewidth=0.9)
    ax.text((x_icam + x_mad) / 2, bracket_y + h * 1.3,
            f"ICAM1 vs MADCAM1\n{pstr}",
            ha="center", va="bottom", fontsize=8,
            style="italic", color=color)

ax.axhline(0, color="black", linewidth=0.8, linestyle="--",
           alpha=0.4)
ax.set_xticks(list(X_POS.values()))
ax.set_xticklabels(
    [f"{resp}\n{gene}" for resp, gene in X_POS.keys()],
    fontsize=9
)
ax.axvline(1.1, color="gray", linestyle="--", linewidth=0.8, alpha=0.7)
ax.set_ylabel("Mean expression (log-normalized)\nper-sample pseudobulk ± SEM")
ax.set_xlabel("")

legend_elements = [
    mpatches.Patch(facecolor="#B71C1C", label=f"ICAM1 {g1}"),
    mpatches.Patch(facecolor="#EF9A9A", label=f"MADCAM1 {g1}"),
    mpatches.Patch(facecolor="#0D47A1", label=f"ICAM1 {g2}"),
    mpatches.Patch(facecolor="#90CAF9", label=f"MADCAM1 {g2}"),
]
ax.legend(handles=legend_elements, fontsize=8,
          loc="upper right", framealpha=0.9)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/ICAM1_MADCAM1_endothelial_barplot.png",
            dpi=150, bbox_inches="tight")
plt.savefig(f"{OUTPUT_DIR}/ICAM1_MADCAM1_endothelial_barplot.pdf",
            bbox_inches="tight")
plt.show()
plt.close()
print("Saved: ICAM1_MADCAM1_endothelial_barplot.png/.pdf")

pb_endo.to_csv(
    f"{OUTPUT_DIR}/ICAM1_MADCAM1_endothelial_per_sample.csv", index=False
)
print("Saved: ICAM1_MADCAM1_endothelial_per_sample.csv")